In [38]:
from google.colab import userdata
GROQ_API_KEY = userdata.get('Groq_key')
GEMINI_API_KEY=userdata.get('Gemini_Api_Key')

In [39]:
!pip install -q langgraph
!pip install -q langchain-groq
!pip install -q langchain
!pip install -q langchain-core langchain-anthropic
!pip install -q langchain-openai
!pip install -q langchain-community
!pip install -q python-dotenv
!pip install -q langchain-google-genai

In [40]:
from typing import TypedDict
import pandas as pd
import numpy as np
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq

In [41]:
MODELS = {
    "insight": {
        "provider": "gemini",
        "model": "gemini-2.5-flash"
    },

    "content": {
        "provider": "groq",
        "model": "llama-3.1-8b-instant"
    }
}

In [42]:
def get_llm(agent_name: str, temperature: float):

    config = MODELS[agent_name]

    if config["provider"] == "gemini":
        return ChatGoogleGenerativeAI(
            model=config["model"],
            google_api_key=GEMINI_API_KEY,
            temperature=temperature,
        )

    elif config["provider"] == "groq":
        return ChatGroq(
            model=config["model"],
            api_key=GROQ_API_KEY,
            temperature=temperature,
        )

    else:
        raise ValueError(f"Unsupported provider: {config['provider']}")

In [43]:
import json

with open("/content/techpulse.json", "r", encoding="utf-8") as file:
    facebook_json= json.load(file)

In [44]:
from typing import TypedDict
import pandas as pd

class CampaignDetails(TypedDict):
    title: str
    description: str
    target_audience: str
    highlights: list[str]
    value_proposition: list[str]
    promotion: str
    additional_notes: str
    campaign_length: int
    campaign_unit: str
    posts_per_week: int

class PostItemDict(TypedDict):
    post_id: int
    day: str
    suggested_time: str
    objective: str
    post_content: str
    reasoning: str
    status: str

class ContentPlanDict(TypedDict):
    campaign_summary: str
    posts: list[PostItemDict]

class MarketingState(TypedDict):
    raw_data: dict
    dataframe: pd.DataFrame
    insights: str
    campaign_details: CampaignDetails
    content_plan: ContentPlanDict
    # --- Human-in-the-loop fields ---
    human_feedback: str          # free-text feedback from the reviewer
    review_action: str           # "approve" | "regenerate_all" | "regenerate_post"
    target_post_id: int          # which post_id to regenerate (if action == regenerate_post)
    feedback_history: list       # log of every round of feedback, for traceability

In [ ]:
import pandas as pd

initial_state = {
    "raw_data": facebook_json,
    "dataframe": [],  # list of records, not a raw DataFrame (checkpointer needs JSON-safe state)
    "insights": "",
    "campaign_details": {
        "title": "",
        "description": "",
        "target_audience": "",
        "promotion": "",
        "campaign_length": 0,
        "campaign_unit": "",
        "posts_per_week": 0
    },
    "content_plan": {
        "campaign_summary": "",
        "posts": []
    },
    "human_feedback": "",
    "review_action": "",
    "target_post_id": None,
    "feedback_history": []
}

# **Data Agent**

In [46]:
import pandas as pd

def data_agent(state: MarketingState) -> dict:
    raw = state["raw_data"]
    posts = raw["data"] if isinstance(raw, dict) else raw

    # -----------------------------
    # 1. Flatten Facebook JSON -> DataFrame
    # -----------------------------
    rows = []
    for post in posts:
        insights_list = post.get("insights", {}).get("data", [])
        insights_map = {item["name"]: item["values"][0]["value"] for item in insights_list}

        reach = insights_map.get("post_impressions_fan", 0)
        clicks = insights_map.get("post_clicks", 0)
        reactions_by_type = insights_map.get("post_reactions_by_type_total", {})

        rows.append({
            "message": post.get("message", ""),
            "created_time": post.get("created_time", "Unknown"),
            "reach": reach,
            "clicks": clicks,
            "reactions": post.get("reactions", {}).get("summary", {}).get("total_count", 0),
            "comments": post.get("comments", {}).get("summary", {}).get("total_count", 0),
            "shares": post.get("shares", {}).get("count", 0)
        })

    df = pd.DataFrame(rows).drop_duplicates()

    numeric_columns = df.select_dtypes(include="number").columns
    df[numeric_columns] = df[numeric_columns].fillna(0)
    df = df.fillna("Unknown")

    # -----------------------------
    # 2. Feature Engineering (Engagement Metrics)
    # -----------------------------
    df["total_engagement"] = df["reactions"] + df["comments"] + df["shares"]
    df["engagement_rate"] = df.apply(
        lambda r: round(r["total_engagement"] / r["reach"], 4) if r["reach"] > 0 else 0, axis=1
    )
    df["ctr"] = df.apply(
        lambda r: round(r["clicks"] / r["reach"], 4) if r["reach"] > 0 else 0, axis=1
    )
    # -----------------------------
    # 5. Return Cleaned State
    # -----------------------------
    return {
        # store as list of dict records so LangGraph's checkpointer (msgpack)
        # can serialize the state - reconstruct with pd.DataFrame(...) when needed
        "dataframe": df.to_dict("records")
    }

In [47]:
df = pd.DataFrame(data_agent(initial_state)["dataframe"])
df.tail()


,message,created_time,reach,clicks,reactions,comments,shares,total_engagement,engagement_rate,ctr
95,Sick of scrolling job listings that all want s...,2026-06-17T11:15:00+0000,1996,112,93,17,22,132,0.0661,0.0561
96,One of our graduates from the AI and Machine L...,2026-06-19T15:30:00+0000,4286,208,192,20,15,227,0.0530,0.0485
97,Wondering if it's too late to start over in a ...,2026-06-22T12:15:00+0000,2110,180,88,6,19,113,0.0536,0.0853
98,Wondering if it's too late to start over in a ...,2026-06-24T16:15:00+0000,4711,389,357,59,76,492,0.1044,0.0826
99,Stuck in a job that stopped teaching you anyth...,2026-06-30T20:30:00+0000,7565,644,540,83,58,681,0.0900,0.0851


# **Insights Agent**

In [48]:
INSIGHT_AGENT_PROMPT = """You are a senior Marketing Insights Analyst.

Your task is to analyze Facebook posts data and identify patterns that can improve future marketing content.

The data has already been cleaned and contains the following information for each post:
- message
- created time
- reach
- clicks
- reactions
- comments
- shares
- total engagement
- engagement rate
- ctr

Here is the data:
{dataset}

Analyze the posts and provide actionable marketing insights.

Focus on:
1. What time of day and day of week do the highest-performing posts (by engagement rate and ctr) tend to be published at? Recommend the best posting windows based only on what the data shows.
2. Which specific keywords, phrases, or opening hooks appear repeatedly in the highest-performing posts?
3. Which posts achieved the highest reach, and what do they have in common?
4. Which posts achieved the highest engagement rate, and what do they have in common?
5. What writing style seems most effective (tone, length, structure, use of questions, offers, or urgency)?
6. What common characteristics appear across successful posts overall?
7. Any noticeable trends connecting post content or timing to performance.
8. Specific, practical recommendations the Content Agent can follow to write a better future post.

Use only the metrics and text provided. Do not invent numbers, dates, or patterns that are not
actually supported by the data. If a point cannot be answered confidently from the data given
(for example, not enough posts to detect a timing pattern), say so briefly instead of guessing.

Return your answer as clear bullet points grouped under these exact headings:

Best Posting Times
Content Tips
Top Keywords And Hooks
Success Characteristics
Trends
"""

In [49]:
from langchain_core.messages import HumanMessage

def insight_agent(state: MarketingState) -> dict:

    df = pd.DataFrame(state["dataframe"])

    analysis_df = df[
        [
            "message",
            "created_time",
            "reach",
            "clicks",
            "reactions",
            "comments",
            "shares",
            "total_engagement",
            "engagement_rate",
            "ctr",
        ]
    ]

    llm = get_llm("insight", temperature=0.3)

    prompt = INSIGHT_AGENT_PROMPT.format(
        dataset=analysis_df.to_markdown(index=False)
    )

    response = llm.invoke([HumanMessage(content=prompt)])

    return {
        "insights": response.content
    }

# **Content Agent**

In [50]:
from pydantic import BaseModel, Field
from typing import Literal

class PostItem(BaseModel):
    day: str = Field(description="Day name, e.g. Monday")
    suggested_time: str = Field(description="Suggested posting time, e.g. 7:00 PM")
    objective: str = Field(description="Post objective: Awareness / Engagement / Conversion")
    post_content: str = Field(description="Full post content, ready to publish")
    reasoning: str = Field(description="Reason for choosing this time and content, based on the insights")

class ContentPlan(BaseModel):
    campaign_summary: str = Field(description="Short summary of the campaign strategy")
    posts: list[PostItem]

In [ ]:
CONTENT_AGENT_PROMPT = """
You are an expert Facebook Marketing Content Strategist.

Your task is to create a complete Facebook content plan for promoting whatever campaign is
described below. The campaign can be for any kind of page, brand, product, or service - do not
assume it is a course, a training program, or any specific industry unless the campaign details
say so explicitly. Base everything strictly on the details provided for THIS campaign.

You will receive:

1. Marketing insights extracted from previous successful Facebook posts (for this same page):
{insights}

2. Details about what is being promoted in this campaign:
{offer_details}

3. The requested campaign details (duration, number of posts, or posting frequency):
{campaign_details}

Your objective is to generate a content calendar that maximizes engagement while following the
successful patterns identified in the marketing insights above, and staying true to whatever is
described in the campaign details (product, service, event, cause, etc).

Guidelines:

- Base your writing style, hooks, keywords, and posting times on the marketing insights provided.
  Do not ignore them or default to generic advertising style.
- Ground every post in the specific campaign details given (its own title, description, audience,
  promotion, and campaign context). Do not invent facts, features, or claims that were not
  that were not provided.
- Create Facebook posts according to the schedule requested by the user.
- Every post should have a different objective (awareness, value, social proof, urgency, enrollment,
  FAQ, etc.), and no two posts should share the same objective unless the campaign length requires it.
- Avoid repeating the same wording, opening hook, or sentence structure across posts.
- Do not mention any price or cost.
- Do not use emojis, hashtags, or any wording that reads as AI generated or templated.
- Keep the posts engaging and persuasive, matching the tone of the successful posts described in the
  insights.
- Include a clear Call-To-Action in every post.
- Recommend an appropriate publishing time for every post, based on the best posting times identified
  in the insights. If the insights don't support a strong timing pattern, use reasonable general
  best practice instead and note that assumption.
- Distribute the posts evenly across the requested campaign duration.
- Adapt the number of posts to the campaign duration if an exact number wasn't specified.

For every scheduled post, return an object with exactly these fields:
- day
- suggested_time
- objective
- post_content
- reasoning (a short explanation of why this post fits the strategy, referencing the specific
  insight or pattern it is based on)

STRICT REQUIREMENTS:
- The number of generated posts MUST exactly match the number requested by the user.
- Do NOT add extra posts or omit any posts.
- Follow the requested campaign duration and posting frequency exactly as provided.
- These constraints are mandatory and must not be modified or inferred differently.

Return STRICT JSON only, with this exact shape, and nothing else before or after it (no markdown
code fences, no explanation):

{{
  "campaign_summary": "...",
  "posts": [
    {{
      "day": "...",
      "suggested_time": "...",
      "objective": "...",
      "post_content": "...",
      "reasoning": "..."
    }}
  ]
}}
"""


In [52]:
import time

QUOTA_MARKERS = ["429", "rate_limit", "rate limit", "quota", "insufficient_quota", "too many requests"]

def classify_llm_error(e: Exception) -> str:
    """Best-effort read of the exception to tell a real quota/rate-limit
    problem apart from other failures (bad JSON, schema mismatch, etc)."""
    msg = str(e).lower()
    if any(marker in msg for marker in QUOTA_MARKERS):
        return "quota_or_rate_limit"
    return "other"

def friendly_error_message(error_type: str, raw_error: str) -> str:
    if error_type == "quota_or_rate_limit":
        return (
            "\u26a0\ufe0f It looks like the model's quota/rate limit (Groq) has been hit or exhausted.\n"
            "Check usage/limits at console.groq.com. Retrying automatically won't help here until"
            " the quota resets or you switch models."
        )
    return (
        "\u26a0\ufe0f The model failed to produce a response in the expected format (not necessarily a quota issue).\n"
        f"Error details: {raw_error[:300]}"
    )

def call_structured_llm_with_retry(structured_llm, prompt, max_attempts=3, backoff_seconds=2):
    """Tries the structured-output call up to max_attempts times.
    Returns (result, error_type, raw_error) - result is None if every attempt failed."""
    last_error = None
    last_error_type = None
    for attempt in range(1, max_attempts + 1):
        try:
            result = structured_llm.invoke([HumanMessage(content=prompt)])
            return result, None, None
        except Exception as e:
            last_error = str(e)
            last_error_type = classify_llm_error(e)
            print(f"  ...attempt {attempt}/{max_attempts} failed ({last_error_type}): {last_error[:120]}")
            if last_error_type == "quota_or_rate_limit":
                # no point burning through retries against an exhausted quota
                break
            if attempt < max_attempts:
                time.sleep(backoff_seconds)
    return None, last_error_type, last_error


In [ ]:
def build_offer_details(campaign: CampaignDetails) -> str:
    return f"""Title: {campaign["title"]}
Description: {campaign["description"]}
Target Audience: {campaign["target_audience"]}
Promotion: {campaign["promotion"]}"""


def content_agent(state: MarketingState) -> dict:
    llm = get_llm("content", temperature=0.8)
    structured_llm = llm.with_structured_output(ContentPlan)

    insights = state["insights"]
    campaign = state["campaign_details"]

    offer_details = build_offer_details(campaign)

    campaign_details = f"""Campaign Length: {campaign["campaign_length"]} {campaign["campaign_unit"]}
Posts Per Week: {campaign["posts_per_week"]}"""

    prompt = CONTENT_AGENT_PROMPT.format(
        insights=insights,
        offer_details=offer_details,
        campaign_details=campaign_details,
    )

    # --- Human-in-the-loop: fold in feedback from a previous review round ---
    feedback = state.get("human_feedback", "")
    if feedback:
        prompt += f"""

IMPORTANT - Human Feedback on the Previous Draft (this is a revision round):
{feedback}

Revise the ENTIRE content plan so it fully addresses this feedback. Keep everything
that was already working well; change only what the feedback asks you to change.
"""

    result, error_type, raw_error = call_structured_llm_with_retry(structured_llm, prompt)

    if result is not None:
        content_plan = result.model_dump()
    else:
        content_plan = {
            "campaign_summary": "",
            "posts": [],
            "error": raw_error,
            "error_type": error_type,
        }

    for idx, post in enumerate(content_plan["posts"], start=1):
        post["post_id"] = idx
        post["status"] = "draft"

    history = state.get("feedback_history", [])
    if feedback:
        history = history + [{"round": len(history) + 1, "scope": "all", "feedback": feedback}]

    # clear the feedback once it's been applied, so it doesn't leak into
    # a later round where the reviewer didn't say anything new
    return {
        "content_plan": content_plan,
        "human_feedback": "",
        "feedback_history": history,
    }


In [54]:
SINGLE_POST_PROMPT = """
You are revising ONE post inside an existing Facebook content calendar.

Marketing insights:
{insights}

Details about what is being promoted:
{offer_details}

Other posts already in the plan (for context only - do not duplicate their hooks or wording):
{other_posts_summary}

The post being revised:
Day: {day}
Suggested Time: {suggested_time}
Objective: {objective}
Current Content: {current_content}

Human feedback on THIS specific post:
{feedback}

Rewrite ONLY this post so it addresses the feedback. Keep the same day, suggested_time and
objective unless the feedback explicitly asks you to change them. Do not reuse wording, hooks,
or sentence structure from the other posts listed above. No emojis, no hashtags, no mention of
price or cost.

Return an object with exactly these fields: day, suggested_time, objective, post_content, reasoning.
"""


def content_agent_single_post(state: MarketingState) -> dict:
    llm = get_llm("content", temperature=0.8)
    structured_llm = llm.with_structured_output(PostItem)

    campaign = state["campaign_details"]
    content_plan = state["content_plan"]
    target_id = state.get("target_post_id")
    feedback = state.get("human_feedback", "")

    posts = content_plan["posts"]
    target_post = next((p for p in posts if p["post_id"] == target_id), None)

    if target_post is None:
        # nothing to do, target_post_id was invalid - just go back to review
        return {"human_feedback": "", "target_post_id": None}

    other_posts_summary = "\n".join(
        f"- ({p['objective']}) {p['post_content'][:80]}..."
        for p in posts if p["post_id"] != target_id
    )

    prompt = SINGLE_POST_PROMPT.format(
        insights=state["insights"],
        offer_details=build_offer_details(campaign),
        other_posts_summary=other_posts_summary,
        day=target_post["day"],
        suggested_time=target_post["suggested_time"],
        objective=target_post["objective"],
        current_content=target_post["post_content"],
        feedback=feedback,
    )

    result, error_type, raw_error = call_structured_llm_with_retry(structured_llm, prompt)

    if result is not None:
        updated = result.model_dump()
        updated["post_id"] = target_id
        updated["status"] = "draft"
        posts = [updated if p["post_id"] == target_id else p for p in posts]
        content_plan["posts"] = posts
        content_plan.pop("error", None)
        content_plan.pop("error_type", None)
    else:
        # keep the old post untouched, but surface the error so human_review can warn about it
        content_plan["error"] = raw_error
        content_plan["error_type"] = error_type

    history = state.get("feedback_history", [])
    history = history + [{"round": len(history) + 1, "scope": f"post_{target_id}", "feedback": feedback}]

    return {
        "content_plan": content_plan,
        "human_feedback": "",
        "target_post_id": None,
        "feedback_history": history,
    }


In [55]:
from langgraph.types import interrupt


def human_review(state: MarketingState) -> dict:
    content_plan = state["content_plan"]

    # This pauses the graph. Whatever dict is passed to interrupt() is what
    # shows up in result["__interrupt__"][0].value on the caller side.
    decision = interrupt({
        "review_type": "content_plan",
        "campaign_summary": content_plan.get("campaign_summary", ""),
        "posts": content_plan.get("posts", []),
        "error": content_plan.get("error"),
        "error_type": content_plan.get("error_type"),
    })

    action = decision.get("action", "approve")

    if action == "regenerate_all":
        return {
            "review_action": "regenerate_all",
            "human_feedback": decision.get("feedback", ""),
        }
    elif action == "regenerate_post":
        return {
            "review_action": "regenerate_post",
            "human_feedback": decision.get("feedback", ""),
            "target_post_id": decision.get("post_id"),
        }
    else:
        return {"review_action": "approve"}


def route_after_review(state: MarketingState) -> str:
    action = state.get("review_action", "approve")
    if action == "regenerate_all":
        return "content_agent"
    elif action == "regenerate_post":
        return "content_agent_single_post"
    return "end"


In [56]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

builder = StateGraph(MarketingState)

builder.add_node("data_agent", data_agent)
builder.add_node("insight_agent", insight_agent)
builder.add_node("content_agent", content_agent)
builder.add_node("content_agent_single_post", content_agent_single_post)
builder.add_node("human_review", human_review)

builder.add_edge(START, "data_agent")
builder.add_edge("data_agent", "insight_agent")
builder.add_edge("insight_agent", "content_agent")
builder.add_edge("content_agent", "human_review")
builder.add_edge("content_agent_single_post", "human_review")

builder.add_conditional_edges(
    "human_review",
    route_after_review,
    {
        "end": END,
        "content_agent": "content_agent",
        "content_agent_single_post": "content_agent_single_post",
    },
)

# A checkpointer is REQUIRED for interrupt()/Command(resume=...) to work -
# it's what lets the graph "remember" where it paused between calls.
checkpointer = MemorySaver()
graph = builder.compile(checkpointer=checkpointer)


In [ ]:
import json
import pandas as pd
import numpy as np

# 1. Load the real JSON
import json

with open("/content/techpulse.json", "r", encoding="utf-8") as file:
    facebook_json= json.load(file)

# 2. Set up the initial_state (Include real campaign data to properly test the Content Agent)
initial_state = {
"raw_data": facebook_json,

"dataframe": [],  # list of records, not a raw DataFrame (checkpointer needs JSON-safe state)

"insights": "",

"campaign_details": {
"title": "Python for Data Science Bootcamp",

"description": "An intensive training program to teach Python and data analysis from beginner to expert",

"target_audience": "Computer Science and Engineering graduates and students interested in Data Science",

"promotion": "20% Discount for the First 50 Registered Individuals",

"campaign_length": 2,

"campaign_unit": "weeks",

"posts_per_week": 3
},

"content_plan": {
"campaign_summary": "",

"posts": []

},
"human_feedback": "",
"review_action": "",
"target_post_id": None,
"feedback_history": []
}

# 3. Run the graph up to the first human checkpoint
# thread_id identifies this run so the checkpointer can pause/resume it
config = {"configurable": {"thread_id": "campaign-run-1"}}
result = graph.invoke(initial_state, config=config)

# **Human Review Loop (Human-in-the-Loop)**

In [58]:
from langgraph.types import Command

def print_content_plan(payload):
    print("Campaign Summary:", payload["campaign_summary"])
    print("-" * 50)
    for p in payload["posts"]:
        print(f"[Post {p['post_id']}] {p['day']} @ {p['suggested_time']} - {p['objective']}")
        print(p["post_content"])
        print()

def ask_for_decision(payload):
    """Collects a reviewer decision from the notebook itself (input()).
    Swap this out for a real UI/API call in production - it only needs
    to return one of the three dict shapes below (or None to stop)."""

    # --- Hard stop: never let a broken/empty plan be silently approved ---
    if payload.get("error"):
        print("=" * 50)
        print(friendly_error_message(payload.get("error_type"), payload.get("error") or ""))
        print("=" * 50)
        choice = input("What would you like to do? (r = retry / s = stop the loop here): ").strip().lower()
        if choice == "s":
            return None  # signals the loop below to stop without approving anything
        # retry with the same inputs (empty feedback = "try again as-is")
        return {"action": "regenerate_all", "feedback": ""}

    print_content_plan(payload)
    approve = input("Approve this content plan as-is? (y/n): ").strip().lower()
    if approve == "y":
        return {"action": "approve"}

    scope = input("Regenerate the (a)ll plan or a (s)ingle post? ").strip().lower()
    feedback_text = input("What should change? Describe your feedback: ").strip()

    if scope == "a":
        return {"action": "regenerate_all", "feedback": feedback_text}
    else:
        post_id = int(input("Which post_id should be regenerated? "))
        return {"action": "regenerate_post", "post_id": post_id, "feedback": feedback_text}


# Keep resuming the graph every time it pauses at human_review, until approved
# (or until the reviewer explicitly stops after an unrecoverable error)
stopped_due_to_error = False
while "__interrupt__" in result:
    interrupt_payload = result["__interrupt__"][0].value
    decision = ask_for_decision(interrupt_payload)
    if decision is None:
        stopped_due_to_error = True
        break
    result = graph.invoke(Command(resume=decision), config=config)

if stopped_due_to_error:
    print("\n\u26d4 Stopped before approving any plan - check the quota/model and try again.")
else:
    print("\n✅ Content plan approved. Pipeline finished.")


Campaign Summary: Python for Data Science Bootcamp Campaign
--------------------------------------------------
[Post 1] Monday @ 14:00-16:00 - Awareness
Ever feel like everyone around you is switching careers except you? Imagine starting your first day in a completely new career, with skills in high demand. Our Python for Data Science Bootcamp is here to help you get started. Learn from expert trainers, work on real-world projects, and get an accredited certificate. Limited spots available! Register now and get a 20% discount on your registration.

[Post 2] Wednesday @ 17:00-19:00 - Value
Our Python for Data Science Bootcamp is an intensive training program that teaches Python and data analysis from beginner to expert. With live sessions, real-world practical projects, and an accredited certificate, you'll gain the skills and confidence to succeed in the field. Don't just take our word for it - most of our graduates apply for their first tech role within a month. Register now and take 

In [66]:
df = pd.DataFrame(result["dataframe"])
print("=" * 50)
print("DATA AGENT OUTPUT")
print("=" * 50)
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head(10))

expected_cols = {"post_id", "message", "created_time", "reach", "clicks",
                  "reactions", "comments", "shares", "total_engagement",
                  "engagement_rate", "ctr"}
missing = expected_cols - set(df.columns)
print("Missing columns:", missing if missing else "None ✅")
print("Nulls:\n", df.isnull().sum())

DATA AGENT OUTPUT
Shape: (100, 10)
Columns: ['message', 'created_time', 'reach', 'clicks', 'reactions', 'comments', 'shares', 'total_engagement', 'engagement_rate', 'ctr']


,message,created_time,reach,clicks,reactions,comments,shares,total_engagement,engagement_rate,ctr
0,We got an update this week from a Cybersecurit...,2026-01-05T18:15:00+0000,3271,98,114,16,28,158,0.0483,0.0300
1,We've had students go from zero technical back...,2026-01-05T21:15:00+0000,10838,453,364,58,37,459,0.0424,0.0418
2,What would change for you if you had a portfol...,2026-01-06T10:15:00+0000,6002,473,193,37,39,269,0.0448,0.0788
3,Been meaning to learn a new skill for months b...,2026-01-13T15:00:00+0000,7214,295,390,77,86,553,0.0767,0.0409
4,Some people spend four years and a lot of mone...,2026-01-15T18:45:00+0000,1244,45,79,12,12,103,0.0828,0.0362
5,One of our graduates from the Full-Stack Web D...,2026-01-17T17:30:00+0000,1955,157,107,20,22,149,0.0762,0.0803
6,Picture yourself submitting your first project...,2026-01-20T15:45:00+0000,3659,116,103,11,15,129,0.0353,0.0317
7,Ever feel like everyone around you is switchin...,2026-01-21T17:45:00+0000,5135,346,205,29,43,277,0.0539,0.0674
8,Sick of scrolling job listings that all want s...,2026-01-26T12:15:00+0000,929,29,38,2,8,48,0.0517,0.0312
9,A good number of people in this program had ne...,2026-01-27T10:00:00+0000,4107,224,314,43,42,399,0.0972,0.0545


Missing columns: {'post_id'}
Nulls:
 message             0
created_time        0
reach               0
clicks              0
reactions           0
comments            0
shares              0
total_engagement    0
engagement_rate     0
ctr                 0
dtype: int64


In [67]:
print("=" * 50)
print("INSIGHT AGENT OUTPUT")
print("=" * 50)
print(result["insights"])
print("\nLength:", len(result["insights"]))

INSIGHT AGENT OUTPUT
Here's an analysis of your Facebook posts data, identifying patterns and providing actionable marketing insights:

### Best Posting Times

*   **Engagement Rate (Reactions, Comments, Shares):** Posts with the highest engagement rates tend to be published on **Saturdays (late morning to early afternoon, e.g., 11:00-14:00)** and **Friday evenings (around 19:00-21:00)**. There's also a notable cluster on **Wednesdays in the afternoon/evening (14:00-20:00)**.
*   **Click-Through Rate (CTR):** Posts with the highest CTRs show a strong tendency towards **mid-week (Tuesday, Wednesday, Thursday) in the late afternoon to evening (17:00-21:00)**. Tuesday evenings, in particular, appear to be a strong window for driving clicks.
*   **Recommendation:**
    *   For content designed to maximize overall interaction (likes, comments, shares), target **weekend mornings/early afternoons (Saturday 11:00-14:00)** or **Friday evenings (19:00-21:00)**.
    *   For content focused on dri

In [68]:
print("=" * 50)
print("CONTENT AGENT OUTPUT")
print("=" * 50)

content_plan = result["content_plan"]

# make sure no error occurred
assert "error" not in content_plan, f"❌ Error: {content_plan.get('error')}"
print("✅ No parsing/schema errors")

# check the overall structure
assert "campaign_summary" in content_plan, "❌ campaign_summary missing"
assert "posts" in content_plan, "❌ posts missing"
assert isinstance(content_plan["posts"], list), "❌ posts should be a list"
assert len(content_plan["posts"]) > 0, "❌ posts list is empty"
print(f"✅ Structure OK — {len(content_plan['posts'])} posts generated")

print("\n--- Campaign Summary ---")
print(content_plan["campaign_summary"])

CONTENT AGENT OUTPUT
✅ No parsing/schema errors
✅ Structure OK — 6 posts generated

--- Campaign Summary ---
Promote our 2-week Python for Data Science Bootcamp with a 20% discount for the first 50 registered individuals, targeting computer science and engineering graduates and students interested in Data Science, with live sessions, real-world practical projects, and an accredited certificate, focusing on faster job opportunities and in-demand skills.


In [69]:
expected_posts_count = (
    initial_state["campaign_details"]["campaign_length"]
    * initial_state["campaign_details"]["posts_per_week"]
)
actual_posts_count = len(content_plan["posts"])

print(f"Expected posts: {expected_posts_count} | Actual posts: {actual_posts_count}")
if expected_posts_count == actual_posts_count:
    print("✅ Post count matches")
else:
    print("⚠️ Post count mismatch — check the prompt, the model is not following the requested count")

Expected posts: 6 | Actual posts: 6
✅ Post count matches


In [70]:
posts_df = pd.DataFrame(content_plan["posts"])
display(posts_df[["day", "suggested_time", "objective", "post_content"]])

,day,suggested_time,objective,post_content
0,Saturday,11:00-14:00,Awareness,Ever feel like everyone around you is switchin...
1,Tuesday,17:00-21:00,Value,Our Python for Data Science Bootcamp offers li...
2,Wednesday,14:00-20:00,Social Proof,We got an update this week from a graduate who...
3,Friday,19:00-21:00,Urgency,Applications close this Friday for our Python ...
4,Sunday,11:00-14:00,Enrollment,Ready to take the first step towards a career ...
5,Thursday,17:00-21:00,FAQ,Still have questions about our Python for Data...


In [71]:
with open("full_test_output.json", "w", encoding="utf-8") as f:
    json.dump({
        "insights": result["insights"],
        "content_plan": result["content_plan"],
        "feedback_history": result.get("feedback_history", [])
    }, f, ensure_ascii=False, indent=2)

print("✅ Saved to full_test_output.json (including human feedback history)")


✅ Saved to full_test_output.json (including human feedback history)
